# gold_circuit_breaker_check
`gold_vuln_daily_build`(DAILY) 이후, `publish_*` 실행 전에 실행한다(05_Job_Orchestration.md §6.7).
5개 조건이 **모두** 정상일 때만 Publish를 허용한다(03_Gold_Schema.md §3.11). 하나라도 미달이면 **직전 정상본을 유지**하고 이번 배치는 Publish하지 않는다 — Job/Pipeline은 이 노트북의 exit value(`proceed_to_publish`)를 읽어 If Condition으로 `publish_*` 실행 여부를 분기해야 한다.

`run_mode != "DAILY"`(BACKFILL/EVENT)인 경우, Job 배선상 이 노트북 자체가 호출되지 않는 경로이므로 Reconciliation 체크는 해당하지 않음으로 처리한다.

In [ ]:
%run 00_gold_common

In [ ]:
def latest_metric(job_name: str, metric_name: str):
    if not table_exists(TBL["meta_dq_metric"]):
        return None
    rows = (
        spark.table(TBL["meta_dq_metric"])
        .filter(
            (F.col("job_name") == job_name)
            & (F.col("metric_name") == metric_name)
            & (F.col("snapshot_date") == snapshot_date)
        )
        .orderBy(F.col("computed_at").desc())
        .limit(1)
        .collect()
    )
    return rows[0] if rows else None

checks = {}
detail = {}


### 1) Source Coverage — 오늘 EPSS 검증이 성공적으로 돌았는가

In [ ]:
epss_reject_metric = latest_metric("validate_epss", "reject_rate")
checks["source_coverage"] = epss_reject_metric is not None
detail["source_coverage"] = "validate_epss dq metric 존재 여부로 판단" if epss_reject_metric is not None \
    else "오늘자 validate_epss 실행 기록이 없음 (EPSS 소스 장애 가능성, 시나리오⑥)"


### 2) Duplicate Rate — Business Key 중복 0건

In [ ]:
dup_daily = 0
if table_exists(TBL["gold_vuln_daily"]):
    dup_daily = (
        spark.table(TBL["gold_vuln_daily"]).filter(F.col("date_key") == snapshot_date)
        .groupBy("cve_id", "date_key").count().filter(F.col("count") > 1).count()
    )

dup_package = 0
if table_exists(TBL["gold_vuln_package"]):
    dup_package = (
        spark.table(TBL["gold_vuln_package"])
        .groupBy("vuln_package_key").count().filter(F.col("count") > 1).count()
    )

checks["duplicate_rate"] = (dup_daily + dup_package) <= DUPLICATE_RATE_MAX
detail["duplicate_rate"] = f"gold_vuln_daily dup={dup_daily}, gold_vuln_package dup={dup_package}"


### 3) Join Match Rate — active_cve_set 대비 오늘 EPSS 매치율 90% 이상

In [ ]:
active_count = spark.table(TBL["active_cve_set"]).count() if table_exists(TBL["active_cve_set"]) else 0
epss_today_count = (
    spark.table(TBL["epss"]).filter(F.col("snapshot_date") == snapshot_date).select("cve_id").distinct().count()
    if table_exists(TBL["epss"]) else 0
)
join_match_rate = (epss_today_count / active_count) if active_count > 0 else 0.0

checks["join_match_rate"] = join_match_rate >= JOIN_MATCH_RATE_THRESHOLD
detail["join_match_rate"] = f"{join_match_rate:.2%} (threshold={JOIN_MATCH_RATE_THRESHOLD:.0%})"


### 4) Reconciliation — `silver_epss`(오늘) == `gold_vuln_daily`(오늘) 완전 일치

In [ ]:
if run_mode == "DAILY":
    silver_epss_today = spark.table(TBL["epss"]).filter(F.col("snapshot_date") == snapshot_date).count() \
        if table_exists(TBL["epss"]) else 0
    gold_daily_today = spark.table(TBL["gold_vuln_daily"]).filter(F.col("date_key") == snapshot_date).count() \
        if table_exists(TBL["gold_vuln_daily"]) else 0
    checks["reconciliation"] = silver_epss_today == gold_daily_today
    detail["reconciliation"] = f"silver_epss={silver_epss_today}, gold_vuln_daily={gold_daily_today}"
else:
    checks["reconciliation"] = True
    detail["reconciliation"] = f"run_mode={run_mode} -- DAILY 경로에서만 적용, 해당 없음으로 통과 처리"


### 5) Schema Drift — 신규/소실 컬럼 0건

In [ ]:
EXPECTED_GOLD_VULN_DAILY_COLUMNS = {
    "cve_id", "date_key", "epss_score", "epss_percentile", "has_epss",
    "epss_delta_1d", "epss_delta_7d",
    "kev_listed", "kev_date_added", "kev_required_action", "has_kev", "days_since_kev_added",
    "ghsa_ids", "is_systemic_day", "systemic_reason", "coverage_score",
    "source_batch_ids", "batch_id", "ingested_at",
}
actual_cols = set(spark.table(TBL["gold_vuln_daily"]).columns) if table_exists(TBL["gold_vuln_daily"]) else set()
drift_cols = actual_cols.symmetric_difference(EXPECTED_GOLD_VULN_DAILY_COLUMNS) if actual_cols else set()

checks["schema_drift"] = len(drift_cols) <= SCHEMA_DRIFT_MAX_COLUMNS
detail["schema_drift"] = f"drifted_columns={sorted(drift_cols)}" if drift_cols else "no drift"


### 결과 집계 및 Pipeline에 전달

In [ ]:
overall_pass = all(checks.values())

result_row = {
    "batch_id": batch_id,
    "snapshot_date": snapshot_date,
    "proceed_to_publish": overall_pass,
    **{f"check_{k}": v for k, v in checks.items()},
    "detail_json": json.dumps(detail, ensure_ascii=False),
    "computed_at": INGESTED_AT,
}
write_append(spark.createDataFrame([result_row]), TBL["meta_circuit_breaker_run"])

for k, v in checks.items():
    log_dq_metric("gold_circuit_breaker_check", k, v, passed=v, detail=detail.get(k))

log_batch_run("gold_circuit_breaker_check", "SUCCESS" if overall_pass else "FAILED", record_count=1)

result_payload = json.dumps({"proceed_to_publish": overall_pass, "checks": checks, "detail": detail}, ensure_ascii=False)
print(f"[gold_circuit_breaker_check] {result_payload}")

try:
    import notebookutils  # Fabric 런타임에서만 사용 가능
    notebookutils.notebook.exit(result_payload)
except ImportError:
    # 로컬/테스트 환경 -- Pipeline이 없으므로 print만 하고, 강제 실패는 시키지 않는다.
    pass
